In [1]:
import ollama
import os
import requests
from utils.logger import CustomLogger
import time
from datetime import datetime
import uuid

from dotenv import load_dotenv

load_dotenv()

OLLAMA_API_KEY = os.getenv("WEB_SEARCH_API_KEY")

if not OLLAMA_API_KEY:
    raise ValueError("OLLAMA_API_KEY not found in environment variables.")

## Define web-search tool

In [2]:
def web_search(query: str, max_results: int = 1) -> dict:
    """ Perform a web search using the Ollama API.
    
    Args:
        query (str): The search query.
        max_results (int): The maximum number of results to return.
    
    Returns:
        dict: The search results from the API.
    """
    url = "https://ollama.com/api/web_search"
    headers = {
    "Authorization": f"Bearer {OLLAMA_API_KEY}",
    "Content-Type": "application/json"
    }
    # Construct the payload for the API request
    payload = {
        "query": query,
        "limit": max_results
    }
    try:
        # Make the POST request to the Ollama API
        response = requests.post(url, 
                                 headers=headers, 
                                 json=payload)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"An error occurred: {e}")
        return None

### Basic Memory Mechanisms

* **sliding window**: keep only recent messages

* **summary**: use llm to summarize key facts

In [3]:
class SummaryLLM:

    def get_summary(self, state_memory):
        summary_prompt = [
            {"role": "system", "content": "You are a helpful assistant that summarizes the conversation history to keep it concise while retaining important information."},
            {"role": "user", "content": "Summarize the following conversation history in a concise manner, keeping important details:\n" + "\n".join([f"{msg['role']}: {msg['content']}" for msg in state_memory])}
        ]
        response = ollama.chat(
            model="mistral-small3.2:24b",
            messages=summary_prompt,
            options={"temperature": 0}
        )
        summary = response["message"].get("content", "")
        # Token usage metrics
        prompt_tokens = response.prompt_eval_count      # tokens used for the prompt
        completion_tokens = response.eval_count         # tokens generated
        total_tokens = prompt_tokens + completion_tokens
        # Duration metrics
        total_duration = response.total_duration/1000  # total time taken for the model call
        return summary, total_tokens, total_duration
    
class WebContentSummaryLLM:

    def summarize_content(self, query, content):
        summary_prompt = [
            {"role": "system", "content": "You are a helpful assistant that summarizes web search results to extract key information."},
            {"role": "user", "content": f"Summarize the following content in a concise manner, extracting key information relevant to the query: '{query}'\nContent:\n{content}"}
        ]
        response = ollama.chat(
            model="mistral-small3.2:24b",
            messages=summary_prompt,
            options={"temperature": 0}
        )
        summary = response["message"].get("content", "")
        return summary

In [4]:
class StatefullAgent:
    
    AVAILABLE_MEMORY_MODES = ["sliding_window", "summary"]

    def __init__(self, llm_model="mistral-small3.2:24b", system_prompt="", temperature=0, max_iterations=10, memory_mode="summary", window_size=10):

        if system_prompt and not isinstance(system_prompt, str):
            raise ValueError("System message must be a string.")
        if not isinstance(max_iterations, int) or max_iterations <= 0:
            raise ValueError("max_iterations must be a positive integer.")
        if memory_mode not in self.AVAILABLE_MEMORY_MODES:
            raise ValueError(f"Unknown memory_mode '{memory_mode}'. Choose from {self.AVAILABLE_MEMORY_MODES}")
        if memory_mode == "sliding_window":
            assert window_size > 0, "window_size must be a positive integer for sliding_window memory mode."

        session_id = str(uuid.uuid4())
        self.session_id = f"session_{session_id}"
        self.tot_token_session = 0
        self.logger_trace = CustomLogger(filename="agent_trace_logs.jsonl")
        self.logger_trace.info("Agent session started", session_id=self.session_id, model=llm_model, created_at=time.time())
        self.interaction = 0
        self.llm_model = llm_model
        self.system_prompt = system_prompt
        self.temperature = temperature
        self.state_memory = []
        self.max_iterations = max_iterations
        if self.system_prompt:
            self.system_message = {"role": "system", "content": self.system_prompt}
            self.logger_trace.info("Initialized system message", role="system", content=self.system_prompt)
        self.state_memory = [self.system_message]
        self.window_size = window_size
        self.memory_mode = memory_mode
        self.llm_summary = SummaryLLM()
        self.content_summary = WebContentSummaryLLM()

    def __call__(self, user_message):
        try:
            self.interaction += 1
            start_time = time.time()
            start_token_interaction = self.tot_token_session  # track tokens at the start of the interaction for metrics
            self.logger_trace.info("Starting new agent interaction", 
                                   interaction=self.interaction, 
                                   call_start_timestamp=time.time()
                                   )
            # Log the user message
            self.user_message = {"role": "user", "content": user_message}
            self.state_memory.append(self.user_message)
            self.logger_trace.info("Received user message", role=self.user_message.get("role", "user"), content=self.user_message.get("content", ""))
            # Execute the agent reasoning and acting loop
            result = self.execute()
            # Log the final assistant message after execution
            self.assistant_message = {"role": "assistant", "content": result}
            self.state_memory.append(self.assistant_message)
            self.logger_trace.info("Final assistant message after execution", role=self.assistant_message.get("role", "assistant"), content=self.assistant_message.get("content", ""))
            # Metrics
            tokens_used_interaction = self.tot_token_session - start_token_interaction  # tokens used in this interaction
            latency = time.time() - start_time  # total time taken for the interaction
            self.logger_trace.info("Ending agent interaction", 
                                   interaction=self.interaction, 
                                   call_end_timestamp=datetime.utcfromtimestamp(time.time()), 
                                   latency=latency, 
                                   tokens_interaction=tokens_used_interaction)
            self.logger_trace.info("Total tokens used in session", total_tokens=self.tot_token_session)
            return result
        except Exception as e:
            self.logger_trace.error(f"Error during agent call: {e}")
            return "An error occurred while processing your request."

    def get_sliding_window_memory(self):
        """
        Keep only the most recent messages in memory up to the window size, always including the system message.
        """
        self.state_memory = self.state_memory[-self.window_size+1:]
        if self.system_message not in self.state_memory:
            self.state_memory.insert(0, self.system_message)

    def get_summary_memory(self):
        """
        Generate a summary of the conversation history and keep only the system message and the summary in memory.
        """
        summary, total_tokens, total_duration = self.llm_summary.get_summary(state_memory=self.state_memory)
        self.tot_token_session +=  total_tokens
        self.logger_trace.info("Generated conversation summary for memory compression", tokens_used=total_tokens, duration=total_duration)
        self.state_memory = [self.system_message, {"role": "assistant", "content": f"Summary of conversation so far: {summary}"}]

    def get_content_summary(self, content):
        """
        Generate a summary of the given content (e.g., web search results) to keep it concise when feeding back to the model.
        """
        summary = self.content_summary.summarize_content(query=self.user_message.get("content", ""), content=content)
        return summary

    def execute(self):

        iterations = 0

        # The agent will go through a loop of reasoning, acting, and observing until it reaches a final answer or hits the max iterations
        while iterations < self.max_iterations:

            iterations += 1

            # Before each reasoning step, check if we need to compress memory
            if len(self.state_memory) > self.window_size:
                if self.memory_mode == "sliding_window":
                    self.get_sliding_window_memory()
                elif self.memory_mode == "summary":
                    self.get_summary_memory()


            # --- THINK ---
            # The model reasons about what to do next
            response = ollama.chat(
                                model=self.llm_model,
                                messages=self.state_memory,
                                options={"temperature": self.temperature},
                                tools=[web_search]
                                )
            
            # Ollama exposes these directly on the response object
            # DURATION METRICS
            total_duration = response.total_duration # total time taken for the model call
            load_duration = round(response.load_duration/total_duration*100,2)  # time taken to load the model as a percentage of total duration
            prompt_eval_duration = round(response.prompt_eval_duration/total_duration*100,2)  # time taken to evaluate the prompt as a percentage of total duration
            eval_duration = round(response.eval_duration/total_duration*100,2)  # time taken to generate the response as a percentage of total duration
            self.logger_trace.info("LLM response timing", iteration = iterations, total_duration=total_duration, load_duration=load_duration, prompt_eval_duration=prompt_eval_duration, eval_duration=eval_duration)
            # TOKEN USAGE METRICS
            prompt_tokens = response.prompt_eval_count      # tokens used for the prompt
            completion_tokens = response.eval_count         # tokens generated
            total_tokens = prompt_tokens + completion_tokens  # total tokens for this model call
            self.tot_token_session += total_tokens  # track total tokens used in the session
            self.logger_trace.info("LLM response received", iteration = iterations, prompt_tokens=prompt_tokens, completion_tokens=completion_tokens, total_tokens=total_tokens)

            # Log the model's message and any tool calls it intends to make
            message = response["message"]
            # get tool calls if present, otherwise default to empty list
            tool_calls = message.get("tool_calls", [])
            self.logger_trace.info("Model response with potential tool calls",
                              iteration = iterations, 
                              role=message.get("role", "assistant"), 
                              content=message.get("content", ""),
                              tool_calls=tool_calls)

            # --- ACT ---
            # The model decides to call tools or provide a final answer
            if tool_calls:

                # If there are tool calls, we log them and execute them one by one, feeding the observations back to the model
                self.assistant_message = {"role": "assistant", 
                                            "content": message.get("content", ""), 
                                            "tool_calls": tool_calls
                                            }               
                self.state_memory.append(self.assistant_message)

                for call in tool_calls:
                    tool_name = call.function.name  # name attribute of the tool function
                    tool_args = call.function.arguments  # arguments for the tool call
                    self.logger_trace.info("Executing tool call",
                                        iteration = iterations, 
                                        tool_name=tool_name, 
                                        tool_args=tool_args
                                        )

                    # --- OBSERVE ---
                    # Run the tool and feed the result back
                    if tool_name == "web_search":
                        query = tool_args.get("query", "")
                        response = web_search(query=query, max_results=1)
                        observation = "Web search results:\n"
                        # Assuming the response has a 'results' field which is a list of search results
                        for k, res in enumerate(response["results"]):
                            observation += f"Content: {res['content']}\n\n"
                    else:
                        observation = f"Unknown tool: {tool_name}"
                    # Summarize the observation before feeding it back to the model to keep the input concise
                    observation_summary = self.get_content_summary(content=observation)
                    # Feed the summarized observation back into the conversation history
                    self.tool_message = {"role": "tool", "tool_name": tool_name, "content": observation_summary}
                    self.state_memory.append(self.tool_message)
                    self.logger_trace.info("Tool observation and summary", iteration = iterations, tool_name=tool_name, observation=observation_summary)
            else:
                # if no tool calls, model provides the final answer 
                final_answer = message.get("content", "")
                return final_answer

        # max iterations hit without a conclusive answer
        return "I was unable to reach a final answer within the allowed number of steps."

In [5]:
PROMPT_SYSTEM = "You are a helpful assistant getting real-time information and summarizing it. You can use the web search tool to find up-to-date information when needed."

In [6]:
memory_agent = StatefullAgent(system_prompt=PROMPT_SYSTEM)    

In [7]:
response = memory_agent("Is there any news related to Venice Football Team?")

In [8]:
print("Final response:", response)

Final response: ### Summary of Recent News Related to Venezia Football Club

1. **Promotion to Serie A:**
   Venezia Football Club has secured promotion back to Italy's Serie A after a 2-2 draw with Spezia, ensuring an immediate return to the top flight following their relegation last season.

2. **Gianluca Busio's Contribution:**
   USMNT midfielder Gianluca Busio played a crucial role in Venezia's promotion, starting and completing the full 90 minutes in the decisive match against Spezia. He has been a key player for the team throughout the season.

3. **League Standing:**
   Venezia currently leads Serie B by a point over second-placed Frosinone and holds a four-point advantage over third-placed Monza. The team aims to finish the season in first place and win the league.

4. **Investment and Ownership:**
   Canadian rapper Drake, who co-owns Venezia FC, has helped secure new investments for the club. An investment vehicle led by Tim Leiweke and his daughter Francesca Bodie contribut

In [9]:
response = memory_agent("and what about Inter?")

In [10]:
print("Final response:", response)

Final response: I'm sorry, but I currently don't have the tools to provide information about Inter. Is there something else I can assist you with?
